# H2 vs Kr Proton-Impact Ionisation Cross-Section Comparison

Loads tabulated cross sections, plots sigma(E), and highlights the 30 keV point.


In [ ]:
from pathlib import Path
import os, sys, subprocess, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Locate project root
_ROOT = Path.cwd()
while _ROOT.name != 'plasma_column' and _ROOT.parent != _ROOT:
    _ROOT = _ROOT.parent
if str(_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(_ROOT / 'src'))

WORK           = Path.home() / 'Work' / 'simulation_codes-working'
WARPX_DATA_DIR = WORK / 'warpx-data'
RESULTS_DIR    = _ROOT / 'results'  # Processed diagnostic CSV/JSON results
RUNS_DIR       = _ROOT / 'runs'     # Raw simulation output directories (gitignored)
PLOTS_DIR      = _ROOT / 'plots'    # Generated publication figures
RESULTS_DIR.mkdir(exist_ok=True)
RUNS_DIR.mkdir(exist_ok=True)
PLOTS_DIR.mkdir(exist_ok=True)

os.environ['WARPX_DATA_DIR']  = str(WARPX_DATA_DIR)
os.environ['LD_LIBRARY_PATH'] = (
    str(WORK / 'warpx' / 'install' / 'lib') + ':'
    + os.environ.get('LD_LIBRARY_PATH', '')
)
print('Python :', sys.executable)
print('ROOT   :', _ROOT)
print('WarpX data:', WARPX_DATA_DIR)


In [ ]:
from plasma_column.notebook_utils import print_simulation_config
_DEFAULTS = {
    'beam energy [keV]':   30.0,
    'cross-section source':'warpx-data/MCC_cross_sections/',
    'gases':               'H2, Kr',
    'energy range [keV]':  '1 - 1000',
}
print_simulation_config(
    notebook_title='H2 vs Kr Cross-Section Comparison',
    defaults=_DEFAULTS, overrides={},
)


## 1. Plot sigma(E)


In [ ]:
from plasma_column.gas import load_cross_section_table, get_h2_cross_section, get_kr_cross_section
from plasma_column.plotting import setup_publication_style, save_figure
setup_publication_style()

BEAM_KEV = 30.0
XS_ROOT  = WARPX_DATA_DIR / 'MCC_cross_sections'
GAS_META = {
    'H2': {'path': XS_ROOT / 'H2' / 'proton_impact_ionization.dat',
           'color': 'tab:blue',   'label': r'H$_2$'},
    'Kr': {'path': XS_ROOT / 'Kr' / 'proton_impact_ionization.dat',
           'color': 'tab:orange', 'label': 'Kr'},
}

fig, ax = plt.subplots(figsize=(9, 5))
for gas, meta in GAS_META.items():
    if not meta['path'].exists():
        print(f'{gas}: not found — {meta["path"]}')
        continue
    df_xs = load_cross_section_table(meta['path'])
    ax.loglog(df_xs.iloc[:,0] * 1e-3, df_xs.iloc[:,1],
              color=meta['color'], lw=2, label=meta['label'])

ax.axvline(BEAM_KEV, color='gray', lw=1.2, ls='--',
           label=f'{BEAM_KEV:.0f} keV operating point')
ax.set_xlabel('Proton kinetic energy [keV]', fontsize=12)
ax.set_ylabel(r'Cross section [m$^2$]', fontsize=12)
ax.set_title(r'Proton-impact ionisation: H$_2$ vs Kr', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, ls='--', alpha=0.5, which='both')
p, _ = save_figure(fig, PLOTS_DIR / 'h2_kr_cross_sections')
plt.show()
print('Saved:', p.name)


## 2. Operating-point read-out


In [ ]:
s_h2 = get_h2_cross_section(BEAM_KEV)
s_kr = get_kr_cross_section(BEAM_KEV)
print(f'sigma(H2, {BEAM_KEV} keV) = {s_h2:.4e} m2')
print(f'sigma(Kr, {BEAM_KEV} keV) = {s_kr:.4e} m2')
print(f'Ratio sigma_Kr / sigma_H2  = {s_kr/s_h2:.2f}')
